# 04 — Kaplan-Meier Survival Curves

**Goal:** Visualise *when* borrowers default, not just *if* they default.
Kaplan-Meier curves stratified by **Traffic Light risk band** demonstrate
that the model correctly orders borrowers by time-to-default risk.

## Contents
1. Derive loan duration from application data
2. Assign Traffic Light bands using trained model
3. Fit KM curves per risk band
4. Plot survival curves with 95% confidence intervals
5. Log-rank test (statistical significance)
6. Segment analysis: KM by employment type and income band
7. Key insight: median months to default by band

In [ ]:
import sys, warnings, os
from pathlib import Path

_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path().resolve().parent
sys.path.insert(0, str(_root / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from models import DefaultClassifier, KaplanMeierAnalyser, derive_loan_duration
from utils.config import load_config  # side-effect: os.chdir(project_root)

sns.set_theme(style='whitegrid')
cfg = load_config()

# Load trained model
clf = DefaultClassifier.load('outputs/models', cfg)
print('Model loaded. CWD:', os.getcwd())


## 1. Load Data & Derive Loan Duration

In [ ]:
df = pd.read_csv('data/processed/feature_cache/master_features.csv')
df = derive_loan_duration(df)

print(f'Data shape: {df.shape}')
print(f'loan_months stats:\n{df["loan_months"].describe()}')


## 2. Assign Traffic Light Bands

In [ ]:
X = df.reindex(columns=clf.feature_columns, fill_value=0)
bands = clf.predict_traffic_light(X)
df['risk_band']   = bands['risk_band'].values
df['risk_score']  = bands['risk_score'].values

print('Band distribution:')
print(df['risk_band'].value_counts())

## 3 & 4. Kaplan-Meier Curves by Risk Band

In [ ]:
km = KaplanMeierAnalyser(df, duration_col='loan_months', event_col='TARGET')
km.fit_by_group('risk_band')

fig = km.plot(figsize=(11, 6))
os.makedirs('outputs/figures', exist_ok=True)
fig.savefig('outputs/figures/04_km_by_risk_band.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)


## 5. Log-Rank Test (Statistical Significance)

In [ ]:
pairs = [('RED','YELLOW'), ('RED','GREEN'), ('YELLOW','GREEN')]
print('Log-rank p-values (null: identical survival):')
for a, b in pairs:
    p = km.log_rank_pvalue(a, b)
    sig = '*** SIGNIFICANT' if p < 0.05 else 'NOT significant'
    print(f'  {a} vs {b}: p = {p:.4e}  →  {sig}')

## 6. Segment Analysis — KM by Employment Type

In [ ]:
if 'NAME_INCOME_TYPE' in df.columns:
    top_income_types = df['NAME_INCOME_TYPE'].value_counts().head(5).index
    df_emp = df[df['NAME_INCOME_TYPE'].isin(top_income_types)].copy()

    km_emp = KaplanMeierAnalyser(df_emp, duration_col='loan_months', event_col='TARGET')
    km_emp.fit_by_group('NAME_INCOME_TYPE')
    fig2 = km_emp.plot(figsize=(11, 6))
    fig2.suptitle('Kaplan-Meier by Income / Employment Type', fontsize=14, fontweight='bold')
    fig2.savefig('outputs/figures/04_km_by_employment.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig2)
else:
    print('NAME_INCOME_TYPE not in master features — skipping segment analysis.')


## 7. Median Survival Times

In [ ]:
median_df = km.median_survival()
print('Median months until default by risk band:')
print(median_df.to_string(index=False))

# Interpretation
print()
print('KEY INSIGHT: RED band borrowers default significantly earlier than')
print('GREEN band borrowers. The Traffic Light system captures time-to-default,')
print('not just binary outcome risk.')